# run_evaluation_v2.ipynb — replicated evaluation with mean ± SE and CIs

Evaluates the runs produced by `run_simulation_v2.ipynb`. Run **once per model family**
(set `MODEL_FAMILY` below to match the simulation notebook).

**Judging protocol (per family):**
1. **Full-simulation judge** — every run × agent judged once (4 variants × 5 reps × 5 agents = 100 calls)
2. **Judge variance** — Baseline replicate 1 re-judged 5× at temperature 0 (25 calls) → isolates judge noise
3. **Per-intervention judge** — one replicate of each variant (4 × 5 agents × 6 events = 120 calls) → appendix tables

**Output — summary tables first, then full results:**
- 6.2.1 Overall Results (per-dimension means with deltas vs Baseline, mean ± SE, 95% CIs on deltas)
- 6.2.3 Model Comparison (Baseline vs Budget)
- 6.2.4 Cost and Latency
- Judge variance (mean ± SE across judge repeats)
- Appendix: full-simulation scores by agent; Baseline scoring by criterion by agent; per-intervention by agent

The Excel export's **first tab (`Report Tables`)** contains the report-formatted 6.2.1 / 6.2.3 / 6.2.4 tables plus simulation-vs-judge variance tables.

All judge scores are **cached to CSV** in `outputs/eval/` — re-running a cell only judges what's missing.

In [ ]:
!pip install -q --disable-pip-version-check --no-warn-script-location \
  anthropic openai \
  'numpy>=1.26' \
  'pandas>=2.2' \
  scipy \
  openpyxl \
  pyyaml

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/Spring2026/fgenai/SIMULATION/berkeley-homes-wildfire-agent-simulation'
except ImportError:
    PROJECT_PATH = os.path.abspath('..')   # running locally: one level up from notebooks/

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import sys
import json
import re
import time
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import yaml
from scipy import stats as st

sys.path.insert(0, PROJECT_PATH)

from src.llm.client import (
    Config, init_clients, init_openrouter_client,
    usage_tracker, judge_intervention, judge_full_simulation,
)

print('Imports OK')

---
## 1. Configuration

In [ ]:
# ── Which family's runs to evaluate (must match run_simulation_v2.ipynb) ──────
MODEL_FAMILY = 'claude'          # 'claude' | 'openai'

# Judge is the opposite family to the simulation models.
JUDGE_MODELS = {'claude': 'openai/gpt-5.4', 'openai': 'claude-opus-4-6'}
JUDGE_MODEL = JUDGE_MODELS[MODEL_FAMILY]
# GPT-5.x reasoning tokens count against max_tokens — give the GPT judge headroom
JUDGE_MAX_TOKENS = 8192 if JUDGE_MODEL.startswith('openai/') else 1024

N_JUDGE_REPEATS   = 5            # judge-variance repeats on Baseline replicate 1
JUDGE_PARALLELISM = 8            # concurrent judge calls

RUNS_DIR       = Path('outputs/runs')
AGENT_YAML_DIR = Path('config/agents/selected')
EVAL_DIR       = Path('outputs/eval')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Judge-score caches — delete a file to force full re-judging of that stage
FULLSIM_CACHE      = EVAL_DIR / f'fullsim_scores_{MODEL_FAMILY}.csv'
JUDGEVAR_CACHE     = EVAL_DIR / f'judge_variance_{MODEL_FAMILY}.csv'
INTERVENTION_CACHE = EVAL_DIR / f'intervention_scores_{MODEL_FAMILY}.csv'

DIMS = ['behavioral_plausibility', 'persona_consistency', 'intervention_responsiveness']
DIM_LABELS = {'behavioral_plausibility': 'Behavioral Plausibility',
              'persona_consistency': 'Persona Consistency',
              'intervention_responsiveness': 'Intervention Responsiveness'}

VARIANT_ORDER = ['Baseline', 'Ablation1_No_Reflection', 'Ablation2_No_Memory_No_Reflection', 'Budget']
DISPLAY_NAMES = {
    'Baseline':                          'Baseline',
    'Ablation1_No_Reflection':           'Ablation 1 (No Reflection)',
    'Ablation2_No_Memory_No_Reflection': 'Ablation 2 (No Memory or Reflection)',
    'Budget':                            'Budget',
}
AGENT_ORDER = ['Laura', 'Linda', 'Walter', 'Margaret', 'Miriam Voss']

print(f'Family: {MODEL_FAMILY} | judge: {JUDGE_MODEL} (temp 0)')

In [ ]:
client_anthropic  = init_clients()
client_openrouter = init_openrouter_client()   # needed whenever the judge is a GPT model

judge_config = Config(JUDGE_MODEL=JUDGE_MODEL, JUDGE_TEMPERATURE=0.0, JUDGE_MAX_TOKENS=JUDGE_MAX_TOKENS)
usage_tracker.reset()

In [ ]:
def load_agent_yaml(yaml_path: Path) -> dict:
    with open(yaml_path, encoding='utf-8') as f:
        raw = yaml.safe_load(f)
    if 'agents' in raw and isinstance(raw['agents'], list):
        return raw['agents'][0]
    return raw

agent_configs = {}
for yaml_path in sorted(AGENT_YAML_DIR.glob('*.yaml')):
    cfg = load_agent_yaml(yaml_path)
    agent_configs[cfg['id']] = cfg
    print(f"  Loaded agent: {cfg['id']} ({cfg['display_name']})")
print(f'\n{len(agent_configs)} agent configs loaded.')

---
## 2. Load simulation runs

In [ ]:
LABEL_RE = re.compile(rf'^{MODEL_FAMILY}_(?P<variant>.+)_rep(?P<rep>\d+)$')

def parse_jsonl(path: Path) -> list:
    entries = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

# {run_label: {variant, rep, run_config, decisions, run_summary, source_file}}
runs = {}
for path in sorted(RUNS_DIR.glob('*.jsonl')):
    entries = parse_jsonl(path)
    run_config = next((e for e in entries if e.get('entry_type') == 'run_config'), None)
    if not run_config:
        continue
    label = run_config.get('run_label', '')
    m = LABEL_RE.match(label)
    if not m:
        continue   # not part of this family's experiment
    run_summary = next((e for e in entries if e.get('entry_type') == 'run_summary'), None)
    if run_summary is None:
        print(f'  SKIP {path.name} ({label}) — incomplete run (no run_summary)')
        continue
    if label in runs:
        prev = RUNS_DIR / runs[label]['source_file']
        if path.stat().st_mtime <= prev.stat().st_mtime:
            print(f'  SKIP {path.name} — older duplicate of {label} (keeping {prev.name})')
            continue
        print(f'  NOTE {label}: replacing {prev.name} with newer {path.name}')
    runs[label] = {
        'variant':     m['variant'],
        'rep':         int(m['rep']),
        'run_id':      path.stem,   # JSONL filename = run_id; keys the judge caches
        'run_config':  run_config,
        'decisions':   [e for e in entries if e.get('entry_type') == 'decision'],
        'run_summary': run_summary,
        'source_file': path.name,
    }

print(f'{len(runs)} completed runs loaded for family {MODEL_FAMILY!r}\n')
_status = pd.DataFrame([{'variant': r['variant'], 'rep': r['rep']} for r in runs.values()])
if not _status.empty:
    print(_status.assign(n=1).pivot_table(index='variant', columns='rep', values='n',
                                          aggfunc='count', fill_value=0).to_string())

def decisions_by_agent(run_data):
    by_agent = defaultdict(list)
    for e in sorted(run_data['decisions'], key=lambda e: e['tick']):
        by_agent[e['agent_id']].append(e)
    return by_agent

def judge_context(agent_id, entries):
    cfg = agent_configs.get(agent_id, {})
    seed_narrative = cfg.get('seed_narrative', entries[0].get('seed_personality', ''))
    memory_seeds   = cfg.get('memory_seeds', [])
    return seed_narrative, memory_seeds

---
## 3. Full-simulation judging — every run × agent, once

100 calls at full coverage. Cached: re-running only judges missing (run_label, agent) pairs.

In [ ]:
def load_cache(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def judge_fullsim_once(label, run_data, agent_id, entries):
    seed_narrative, memory_seeds = judge_context(agent_id, entries)
    all_decisions = [{'day': e['tick'], 'event_type': e['event_type'], 'intervention': e['intervention'],
                      'decision': e['decision'], 'reasoning': e['reasoning']} for e in entries]
    scores = judge_full_simulation(
        client_anthropic=client_anthropic, config=judge_config,
        seed_narrative=seed_narrative, memory_seeds=memory_seeds,
        all_decisions=all_decisions, client_openrouter=client_openrouter,
    )
    return {
        'run_label': label, 'run_id': run_data['run_id'],
        'variant': run_data['variant'], 'rep': run_data['rep'],
        'agent_id': agent_id, 'agent_display_name': entries[0]['agent_display_name'],
        'behavioral_plausibility':     scores.get('behavioral_plausibility'),
        'persona_consistency':         scores.get('persona_consistency'),
        'intervention_responsiveness': scores.get('intervention_responsiveness'),
        'bp_note': scores.get('note_plausibility'),
        'pc_note': scores.get('note_consistency'),
        'ir_note': scores.get('note_responsiveness'),
        'judge_model': JUDGE_MODEL,
    }


def run_judging(jobs, worker, cache_path, key_cols, max_workers=JUDGE_PARALLELISM):
    """Run judge calls in parallel with an on-disk cache. jobs: list of (key_tuple, callable)."""
    cache = load_cache(cache_path)
    if not cache.empty:
        parsed_ok = ~cache[DIMS].apply(pd.to_numeric, errors='coerce').isna().all(axis=1)
        if (~parsed_ok).any():
            print(f'  discarding {(~parsed_ok).sum()} cached rows with unparseable judge output — will re-judge')
            cache = cache[parsed_ok].reset_index(drop=True)
    have = set(map(tuple, cache[key_cols].itertuples(index=False))) if not cache.empty else set()
    todo = [(key, fn) for key, fn in jobs if key not in have]
    print(f'{len(jobs)} judge jobs | {len(jobs) - len(todo)} cached | {len(todo)} to run')
    if not todo:
        return cache
    new_rows, errors = [], []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(fn): key for key, fn in todo}
        for i, future in enumerate(as_completed(futures), 1):
            key = futures[future]
            try:
                new_rows.append(future.result())
            except Exception as exc:
                errors.append((key, repr(exc)))
                print(f'  FAILED {key}: {exc!r}')
            if new_rows and (i % 10 == 0 or i == len(todo)):   # checkpoint the cache
                pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True).to_csv(cache_path, index=False)
            if i % 10 == 0 or i == len(todo):
                print(f'  {i}/{len(todo)} done')
    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True) if new_rows else cache
    cache.to_csv(cache_path, index=False)
    if errors:
        print(f'{len(errors)} failures — re-run this cell to retry them.')
    return cache


fullsim_jobs = []
for label, run_data in sorted(runs.items()):
    for agent_id, entries in decisions_by_agent(run_data).items():
        fullsim_jobs.append(((run_data['run_id'], agent_id),
                             (lambda l=label, rd=run_data, a=agent_id, e=entries:
                              judge_fullsim_once(l, rd, a, e))))

fullsim = run_judging(fullsim_jobs, None, FULLSIM_CACHE, ['run_id', 'agent_id'])
current_run_ids = {r['run_id'] for r in runs.values()}
fullsim = fullsim[fullsim['run_id'].isin(current_run_ids)].reset_index(drop=True)
fullsim[DIMS] = fullsim[DIMS].apply(pd.to_numeric, errors='coerce')
fullsim['overall'] = fullsim[DIMS].mean(axis=1)
_n_partial = int(fullsim[DIMS].isna().any(axis=1).sum())
if _n_partial:
    print(f'WARNING: {_n_partial} rows have missing dimension scores (partial judge parses)')
print(f'\nFull-simulation scores: {len(fullsim)} rows '
      f'({fullsim["variant"].nunique()} variants x {fullsim["rep"].nunique()} reps x '
      f'{fullsim["agent_id"].nunique()} agents)')

---
## 4. Judge variance — Baseline replicate 1 re-judged 5×

Same trajectories, same judge, temperature 0. Any spread here is judge noise (API nondeterminism),
not simulation noise.

In [ ]:
_baseline_reps = sorted(r['rep'] for r in runs.values() if r['variant'] == 'Baseline')
assert _baseline_reps, 'No Baseline runs found — run the simulation notebook first.'
JV_REP = _baseline_reps[0]
jv_label = f'{MODEL_FAMILY}_Baseline_rep{JV_REP}'
jv_run = runs[jv_label]
print(f'Judge-variance target: {jv_label} ({N_JUDGE_REPEATS} repeats x 5 agents)')


def judge_variance_once(repeat, agent_id, entries):
    row = judge_fullsim_once(jv_label, jv_run, agent_id, entries)
    row['repeat'] = repeat
    return row


jv_jobs = []
for repeat in range(1, N_JUDGE_REPEATS + 1):
    for agent_id, entries in decisions_by_agent(jv_run).items():
        jv_jobs.append(((jv_run['run_id'], repeat, agent_id),
                        (lambda r=repeat, a=agent_id, e=entries: judge_variance_once(r, a, e))))

judge_var = run_judging(jv_jobs, None, JUDGEVAR_CACHE, ['run_id', 'repeat', 'agent_id'])
judge_var = judge_var[judge_var['run_id'] == jv_run['run_id']].reset_index(drop=True)
judge_var[DIMS] = judge_var[DIMS].apply(pd.to_numeric, errors='coerce')
judge_var['overall'] = judge_var[DIMS].mean(axis=1)
print(f'Judge-variance scores: {len(judge_var)} rows')

---
## 5. Per-intervention judging — one replicate per variant

Feeds the appendix tables (per-criterion and per-intervention breakdowns).

In [ ]:
def judge_pi_once(label, run_data, entry):
    seed_narrative, memory_seeds = judge_context(entry['agent_id'], [entry])
    scores = judge_intervention(
        client_anthropic=client_anthropic, config=judge_config,
        seed_narrative=seed_narrative, memory_seeds=memory_seeds,
        intervention=entry['intervention'], decision=entry['decision'],
        reasoning=entry['reasoning'], client_openrouter=client_openrouter,
    )
    return {
        'run_label': label, 'run_id': run_data['run_id'],
        'variant': run_data['variant'], 'rep': run_data['rep'],
        'agent_id': entry['agent_id'], 'agent_display_name': entry['agent_display_name'],
        'day': entry['tick'], 'event_type': entry['event_type'],
        'behavioral_plausibility':     scores.get('behavioral_plausibility'),
        'persona_consistency':         scores.get('persona_consistency'),
        'intervention_responsiveness': scores.get('intervention_responsiveness'),
        'bp_note': scores.get('note_plausibility'),
        'pc_note': scores.get('note_consistency'),
        'ir_note': scores.get('note_responsiveness'),
        'judge_model': JUDGE_MODEL,
    }


pi_jobs = []
for variant in VARIANT_ORDER:
    reps = sorted(r['rep'] for r in runs.values() if r['variant'] == variant)
    if not reps:
        print(f'  WARNING: no runs for {variant} — skipping')
        continue
    label = f'{MODEL_FAMILY}_{variant}_rep{reps[0]}'
    run_data = runs[label]
    for entry in sorted(run_data['decisions'], key=lambda e: (e['agent_id'], e['tick'])):
        pi_jobs.append(((run_data['run_id'], entry['agent_id'], entry['tick']),
                        (lambda l=label, rd=run_data, e=entry: judge_pi_once(l, rd, e))))

interventions = run_judging(pi_jobs, None, INTERVENTION_CACHE, ['run_id', 'agent_id', 'day'])
interventions = interventions[interventions['run_id'].isin({r['run_id'] for r in runs.values()})].reset_index(drop=True)
interventions[DIMS] = interventions[DIMS].apply(pd.to_numeric, errors='coerce')
interventions['overall'] = interventions[DIMS].mean(axis=1)
print(f'Per-intervention scores: {len(interventions)} rows')

---
## 6. Statistics helpers

- Replicate-level score = mean over the 5 agents (each agent's score = the full-simulation judge output).
- Mean ± SE per variant = across the 5 replicate-level values.
- Delta CIs vs Baseline use Welch's t (unequal variances).

In [ ]:
def mean_se(values):
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan, np.nan
    se = x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
    return x.mean(), se


def welch_ci(a, b, conf=0.95):
    """CI for mean(a) - mean(b), Welch's t. Returns (delta, lo, hi)."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    d = a.mean() - b.mean()
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    se = np.sqrt(va + vb)
    if se == 0:
        return d, d, d
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    tcrit = st.t.ppf(0.5 + conf / 2, df)
    return d, d - tcrit * se, d + tcrit * se


# Replicate-level scores: one row per (variant, rep), columns = dims + overall
rep_level = (fullsim.groupby(['variant', 'rep'])[DIMS + ['overall']]
             .mean().reset_index())
rep_level['variant'] = pd.Categorical(rep_level['variant'], VARIANT_ORDER, ordered=True)
rep_level = rep_level.sort_values(['variant', 'rep']).reset_index(drop=True)

present_variants = [v for v in VARIANT_ORDER if v in set(rep_level['variant'])]

def _vals(variant, col):
    return rep_level.loc[rep_level['variant'] == variant, col].values

def _model_for(variant):
    for r in runs.values():
        if r['variant'] == variant:
            return r['run_config'].get('decision_model', '?')
    return '?'

# Friendly model names for the report tables (fall back to the raw id if unmapped)
MODEL_DISPLAY = {
    'claude-sonnet-4-6':          'Claude Sonnet 4.6',
    'claude-haiku-4-5-20251001':  'Claude Haiku 4.5',
    'claude-opus-4-6':            'Claude Opus 4.6',
    'openai/gpt-5.4':             'GPT-5.4',
    'openai/gpt-5.4-mini':        'GPT-5.4 mini',
    'openai/gpt-5.4-nano':        'GPT-5.4 nano',
}

def _model_display(variant):
    return MODEL_DISPLAY.get(_model_for(variant), _model_for(variant))

print(rep_level.round(3).to_string(index=False))

---
# SUMMARY TABLES

## 6.2.1 Overall Results

Mean score across 5 agents, averaged over 5 replicate runs. Each agent's complete 60-day trajectory
evaluated in one judge call. Deltas vs Baseline in parentheses.

In [ ]:
base_means = {c: _vals('Baseline', c).mean() for c in DIMS + ['overall']}

rows = []
for v in present_variants:
    row = {'Condition': DISPLAY_NAMES[v]}
    for c in DIMS + ['overall']:
        m = _vals(v, c).mean()
        cell = f'{m:.2f}'
        if v != 'Baseline':
            cell += f' ({m - base_means[c]:+.2f})'
        row[DIM_LABELS.get(c, 'Mean')] = cell
    rows.append(row)
t621 = pd.DataFrame(rows).set_index('Condition')
print('6.2.1 OVERALL RESULTS (mean across replicates; delta vs Baseline in parens)')
print('=' * 100)
print(t621.to_string())

# Mean ± SE companion table
rows = []
for v in present_variants:
    row = {'Condition': DISPLAY_NAMES[v]}
    for c in DIMS + ['overall']:
        m, se = mean_se(_vals(v, c))
        row[DIM_LABELS.get(c, 'Mean')] = f'{m:.2f} \u00b1 {se:.3f}'
    rows.append(row)
print('\n6.2.1b MEAN \u00b1 SE ACROSS 5 REPLICATES (replicate = mean over 5 agents)')
print('=' * 100)
print(pd.DataFrame(rows).set_index('Condition').to_string())

# Delta vs Baseline with 95% CI — the significance test for each ablation effect
rows = []
for v in present_variants:
    if v == 'Baseline':
        continue
    d, lo, hi = welch_ci(_vals(v, 'overall'), _vals('Baseline', 'overall'))
    rows.append({'Condition': DISPLAY_NAMES[v],
                 'Delta (overall, vs Baseline)': f'{d:+.3f}',
                 '95% CI': f'[{lo:+.3f}, {hi:+.3f}]',
                 'CI excludes 0': 'YES — significant' if (lo > 0 or hi < 0) else 'no'})
print('\n6.2.1c DELTA vs BASELINE — 95% CI (Welch t across replicate-level means)')
print('=' * 100)
print(pd.DataFrame(rows).set_index('Condition').to_string())
print('\nA CI that excludes 0 means the ablation effect is distinguishable from run-to-run noise at the 95% level.')

## 6.2.3 Model Comparison

Full-simulation judge scores by dimension for Baseline vs Budget (full system on both, different model tier).

In [ ]:
rows = []
for v in ['Baseline', 'Budget']:
    if v not in present_variants:
        continue
    row = {'Condition': f'{DISPLAY_NAMES[v]} ({_model_for(v)})'}
    for c in DIMS + ['overall']:
        m, se = mean_se(_vals(v, c))
        row[DIM_LABELS.get(c, 'Mean')] = f'{m:.2f} \u00b1 {se:.3f}'
    rows.append(row)
print('6.2.3 MODEL COMPARISON (mean \u00b1 SE across 5 replicates)')
print('=' * 100)
print(pd.DataFrame(rows).set_index('Condition').to_string())

if all(v in present_variants for v in ['Baseline', 'Budget']):
    d, lo, hi = welch_ci(_vals('Budget', 'overall'), _vals('Baseline', 'overall'))
    print(f'\nBudget - Baseline (overall): {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]'
          f' -> {"CI excludes 0" if (lo > 0 or hi < 0) else "CI includes 0"}')

## 6.2.4 Cost and Latency

Quality = full-simulation holistic judge overall score. Agent cost and latency cover decision and
reflection calls only (judge cost excluded), averaged across the 5 replicates.

In [ ]:
rows = []
for v in present_variants:
    summaries = [r['run_summary'] for r in runs.values() if r['variant'] == v]
    q_mean, q_se = mean_se(_vals(v, 'overall'))
    c_mean, c_se = mean_se([s.get('agent_cost_usd') for s in summaries])
    l_mean, l_se = mean_se([s.get('latency_seconds') for s in summaries])
    rows.append({
        'Condition': f'{DISPLAY_NAMES[v]} ({_model_for(v)})',
        'Quality (overall, 1-5)': f'{q_mean:.2f} \u00b1 {q_se:.3f}',
        'Agent cost / run (USD)': f'${c_mean:.2f} \u00b1 {c_se:.3f}',
        'Latency / run (s)':      f'{l_mean:.0f} \u00b1 {l_se:.0f}',
    })
print('6.2.4 COST AND LATENCY (mean \u00b1 SE across 5 replicates)')
print('=' * 110)
print(pd.DataFrame(rows).set_index('Condition').to_string())
print('\nNote: with parallel execution, per-run latency reflects each run\'s own wall-clock, '
      'not total experiment time.')

## Judge variance — same run judged 5×

Baseline replicate 1, identical trajectories, temperature 0. Per repeat we take the mean over agents,
then report mean ± SE across the 5 repeats. Compare the SE here (judge noise) against the SE in 6.2.1b
(simulation noise + judge noise): if they're similar, most of the observed spread is the judge itself.

In [ ]:
per_repeat = judge_var.groupby('repeat')[DIMS + ['overall']].mean()
rows = []
for c in DIMS + ['overall']:
    m, se = mean_se(per_repeat[c])
    rows.append({'Dimension': DIM_LABELS.get(c, 'Overall (mean of 3)'),
                 'Mean': f'{m:.2f}', 'SE (judge)': f'{se:.3f}',
                 'Min repeat': f'{per_repeat[c].min():.2f}', 'Max repeat': f'{per_repeat[c].max():.2f}'})
print(f'JUDGE VARIANCE — {jv_label} judged {N_JUDGE_REPEATS}x by {JUDGE_MODEL} @ temp 0')
print('=' * 90)
print(pd.DataFrame(rows).set_index('Dimension').to_string())

print('\nPer-agent overall score by judge repeat (stability check):')
print(judge_var.pivot_table(index='agent_display_name', columns='repeat', values='overall')
      .round(2).to_string())

sim_se = mean_se(_vals('Baseline', 'overall'))[1]
jud_se = mean_se(per_repeat['overall'])[1]
print(f'\nSE comparison (overall): simulation replicates = {sim_se:.3f} | judge repeats = {jud_se:.3f}')

---
# APPENDIX — Detailed Simulation Results

## A. Full Simulation Scores by Agent

Each cell = overall score (mean of BP, PC, IR) from the full-simulation judge, averaged across the
5 replicate runs. Shows whether ablation effects are consistent across agents or driven by specific profiles.

In [ ]:
agents_present = [a for a in AGENT_ORDER if a in set(fullsim['agent_display_name'])] + \
                 sorted(set(fullsim['agent_display_name']) - set(AGENT_ORDER))

appx_a = (fullsim.pivot_table(index='variant', columns='agent_display_name', values='overall')
          .reindex(present_variants)[agents_present]
          .rename(index=DISPLAY_NAMES))
print('FULL SIMULATION SCORES BY AGENT (overall, mean across 5 replicates)')
print('=' * 100)
print(appx_a.round(2).to_string())

## B. Baseline Variant: Scoring by Criterion by Agent

From the per-intervention judge on the Baseline replicate. Each value = mean of that agent's 6
per-intervention scores on that criterion.

In [ ]:
pi_base = interventions[interventions['variant'] == 'Baseline']
rows = []
for c in DIMS:
    per_agent = pi_base.groupby('agent_display_name')[c].mean()
    rows.append({'Criterion': DIM_LABELS[c], **{a: per_agent.get(a, np.nan) for a in agents_present}})
per_agent_mean = pi_base.groupby('agent_display_name')['overall'].mean()
rows.append({'Criterion': 'Mean', **{a: per_agent_mean.get(a, np.nan) for a in agents_present}})
appx_b = pd.DataFrame(rows).set_index('Criterion')
print(f"BASELINE VARIANT: SCORING BY CRITERION BY AGENT (run: {pi_base['run_label'].iloc[0] if not pi_base.empty else 'n/a'})")
print('=' * 100)
print(appx_b.round(2).to_string())

## C. Baseline Variant: Scoring by Intervention by Agent

One table per criterion: rows = the 6 interventions, columns = agents, plus the agent average.
(Per-intervention scores for the other variants are in the cache CSV / Excel export.)

In [ ]:
def event_label(row):
    return f"Day {row['day']}: {row['event_type']}"

pi_base = pi_base.assign(event=pi_base.apply(event_label, axis=1))
event_order = (pi_base[['day', 'event']].drop_duplicates().sort_values('day')['event'].tolist())

for c in DIMS:
    tbl = (pi_base.pivot_table(index='event', columns='agent_display_name', values=c)
           .reindex(event_order)[agents_present])
    tbl.loc['Agent average'] = tbl.mean()
    print(f'\n{DIM_LABELS[c].upper()}')
    print('=' * 100)
    print(tbl.round(2).to_string())

---
## Export to Excel + judging cost

---
# Report tables (Excel tab 1)

Report-formatted versions of 6.2.1 / 6.2.3 / 6.2.4 (matching the writeup layout, deltas vs
Baseline in parentheses) plus two variance views: run-to-run **simulation** variance and
repeat-judging **judge** variance, and a direct comparison of the two. These are printed below
and written to the **first tab** of the Excel export, formatted with the report's table styling.

In [ ]:
# ── Label + formatting helpers ────────────────────────────────────────────────
def _cond_with_model(v):
    """'Baseline (Claude Sonnet 4.6)' / 'Ablation 1 (No Reflection - Claude Sonnet 4.6)'."""
    disp, model = DISPLAY_NAMES[v], _model_display(v)
    return f'{disp[:-1]} - {model})' if disp.endswith(')') else f'{disp} ({model})'

def _sd(vals):
    v = np.asarray(vals, dtype=float); v = v[~np.isnan(v)]
    return v.std(ddof=1) if len(v) > 1 else np.nan

def _fmt_msd(vals):
    """'mean ± SD' across repeated values."""
    v = np.asarray(vals, dtype=float); v = v[~np.isnan(v)]
    if len(v) == 0:
        return 'n/a'
    if len(v) == 1:
        return f'{v.mean():.2f}'
    return f'{v.mean():.2f} ± {v.std(ddof=1):.3f}'

DIM_COLS = [DIM_LABELS[d] for d in DIMS] + ['Mean']   # header labels for the 4 score columns
_COLS = DIMS + ['overall']                             # matching dataframe columns

# ── 6.2.1 Overall Results (means, deltas vs Baseline) ─────────────────────────
rows = []
for v in present_variants:
    row = {}
    for c, label in zip(_COLS, DIM_COLS):
        m = _vals(v, c).mean()
        row[label] = f'{m:.2f}' if v == 'Baseline' else f'{m:.2f} ({m - base_means[c]:+.2f})'
    rows.append(row)
t_overall = pd.DataFrame(rows, index=[DISPLAY_NAMES[v] for v in present_variants])
t_overall.index.name = 'Condition'

# ── 6.2.3 Model Comparison (Baseline vs Budget, plain means, friendly names) ──
rows, idx = [], []
for v in ['Baseline', 'Budget']:
    if v not in present_variants:
        continue
    rows.append({label: f'{_vals(v, c).mean():.2f}' for c, label in zip(_COLS, DIM_COLS)})
    idx.append(_cond_with_model(v))
t_model = pd.DataFrame(rows, index=idx)
t_model.index.name = 'Condition'

# ── 6.2.4 Cost and Latency ────────────────────────────────────────────────────
rows, idx = [], []
for v in present_variants:
    summaries = [r['run_summary'] for r in runs.values() if r['variant'] == v]
    q    = _vals(v, 'overall').mean()
    cost = np.nanmean([s.get('agent_cost_usd', np.nan) for s in summaries])
    lat  = np.nanmean([s.get('latency_seconds', np.nan) for s in summaries])
    rows.append({'Quality (overall, 1–5)': f'{q:.2f}',
                 'Agent cost / run (USD)': f'${cost:.2f}',
                 'Latency / run (seconds)': f'{lat:.0f}'})
    idx.append(_cond_with_model(v))
t_cost = pd.DataFrame(rows, index=idx)
t_cost.index.name = 'Condition'

# ── Simulation variance: SD across the 5 replicate runs, per condition ────────
rows = []
for v in present_variants:
    rows.append({label: _fmt_msd(_vals(v, c)) for c, label in zip(_COLS, DIM_COLS)})
t_simvar = pd.DataFrame(rows, index=[DISPLAY_NAMES[v] for v in present_variants])
t_simvar.index.name = 'Condition'

# ── Judge variance: Baseline replicate 1 judged N_JUDGE_REPEATS times ─────────
t_judgevar = pd.DataFrame(
    [{label: _fmt_msd(per_repeat[c].values) for c, label in zip(_COLS, DIM_COLS)}],
    index=[f'{DISPLAY_NAMES["Baseline"]} (judged {N_JUDGE_REPEATS}×)'])
t_judgevar.index.name = 'Condition'

# ── Variance comparison: simulation SD vs judge SD (Baseline) ─────────────────
t_varcmp = pd.DataFrame(
    [{label: f'{_sd(_vals("Baseline", c)):.3f}' for c, label in zip(_COLS, DIM_COLS)},
     {label: f'{_sd(per_repeat[c].values):.3f}' for c, label in zip(_COLS, DIM_COLS)}],
    index=['Simulation (5 runs, 1 judge call each)', 'Judge (1 run, 5 judge calls)'])
t_varcmp.index.name = 'Source of variance (SD)'

# ── Assemble (title, dataframe, caption) — order matches the Excel tab ─────────
REPORT_TABLES = [
    ('6.2.1 Overall Results', t_overall,
     'Mean score across 5 agents × 5 replicate runs (25 full-simulation judge calls per '
     'condition). Each agent’s complete 60-day trajectory is evaluated in one judge call, '
     'capturing narrative coherence and cross-event integration. Deltas vs Baseline in parentheses.'),
    ('6.2.3 Model Comparison', t_model,
     'Full-simulation judge scores by dimension (BP, PC, IR) for Baseline vs Budget, means across '
     '5 agents × 5 replicate runs. Shows which dimensions are most affected by model choice.'),
    ('6.2.4 Cost and Latency', t_cost,
     'Quality = full-simulation holistic judge overall score (mean of BP, PC, IR across agents and '
     'replicates). Agent cost and latency cover decision and reflection calls only, averaged across '
     'the 5 replicate runs.'),
    ('Simulation Variance (run-to-run)', t_simvar,
     'Mean ± SD of the per-run score across the 5 replicate runs of each condition (agent-mean '
     'per run). Seed memories and the judge are held fixed, so this SD is run-to-run simulation '
     'stochasticity (decision/reflection sampling).'),
    ('Judge Variance (repeat judging)', t_judgevar,
     f'Baseline replicate 1 re-judged {N_JUDGE_REPEATS}× at temperature 0 — identical '
     'trajectories, same judge. Mean ± SD across the judge repeats (agent-mean per repeat). '
     'Isolates judge (API) nondeterminism.'),
    ('Variance Comparison — Simulation vs Judge (Baseline)', t_varcmp,
     'Standard deviation of the Baseline score from two sources: simulation (5 independent runs) vs '
     'judge (1 run scored 5×). Judge SD ≪ simulation SD means most observed spread is '
     'genuine run-to-run behavioural variation rather than judge noise.'),
]

print('\n' + '#' * 100)
print('REPORT-FORMATTED TABLES  (written to Excel tab 1: "Report Tables")')
print('#' * 100)
for title, df, caption in REPORT_TABLES:
    print(f'\n{title}')
    print('=' * 100)
    print(df.to_string())
    print(f'({caption})')


In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
export_path = EVAL_DIR / f'evaluation_v2_{MODEL_FAMILY}_{timestamp}.xlsx'

cost_rows = []
for label, r in sorted(runs.items()):
    cost_rows.append({'run_label': label, 'variant': r['variant'], 'rep': r['rep'],
                      'agent_model': r['run_config'].get('decision_model'),
                      'agent_cost_usd': r['run_summary'].get('agent_cost_usd'),
                      'latency_seconds': r['run_summary'].get('latency_seconds')})

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

_THIN = Side(style='thin', color='000000')
_BORDER = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)
_HEADER_FILL = PatternFill('solid', fgColor='D9D9D9')

def write_report_table(ws, start_row, title, df, caption, index_header):
    """Write one report-styled table (bold title, grey header, borders, italic caption)."""
    r = start_row
    ws.cell(r, 1, title).font = Font(bold=True, size=12)
    r += 1
    cols = [index_header] + list(df.columns)
    for j, name in enumerate(cols, start=1):
        c = ws.cell(r, j, name)
        c.font = Font(bold=True)
        c.fill = _HEADER_FILL
        c.border = _BORDER
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    r += 1
    for idx, row_vals in zip(df.index, df.itertuples(index=False)):
        c0 = ws.cell(r, 1, str(idx))
        c0.border = _BORDER
        c0.alignment = Alignment(horizontal='left', vertical='center')
        for j, val in enumerate(row_vals, start=2):
            c = ws.cell(r, j, val)
            c.border = _BORDER
            c.alignment = Alignment(horizontal='center', vertical='center')
        r += 1
    cap = ws.cell(r, 1, caption)
    cap.font = Font(italic=True, size=9)
    cap.alignment = Alignment(wrap_text=True, vertical='top')
    r += 1
    # Column widths (widest requirement across all tables wins)
    ws.column_dimensions['A'].width = max(ws.column_dimensions['A'].width or 0, 48)
    for j in range(2, len(cols) + 1):
        L = get_column_letter(j)
        ws.column_dimensions[L].width = max(ws.column_dimensions[L].width or 0, 24)
    return r + 2   # blank gap before the next table

with pd.ExcelWriter(export_path, engine='openpyxl') as writer:
    wb = writer.book
    default_ws = wb.active   # openpyxl's blank default sheet — removed at the end
    ws = wb.create_sheet('Report Tables', 0)   # first tab
    row = 1
    ws.cell(row, 1, f'Evaluation report tables — {MODEL_FAMILY} family  '
                    f'(judge: {JUDGE_MODEL})').font = Font(bold=True, size=14)
    row += 2
    for title, df, caption in REPORT_TABLES:
        row = write_report_table(ws, row, title, df, caption, df.index.name)

    # Raw / detailed sheets after the report tab
    fullsim.to_excel(writer, sheet_name='FullSim Scores', index=False)
    rep_level.to_excel(writer, sheet_name='Replicate Level', index=False)
    judge_var.to_excel(writer, sheet_name='Judge Variance', index=False)
    interventions.to_excel(writer, sheet_name='Per-Intervention', index=False)
    pd.DataFrame(cost_rows).to_excel(writer, sheet_name='Cost & Latency', index=False)
    appx_a.to_excel(writer, sheet_name='Appx A FullSim by Agent')
    appx_b.to_excel(writer, sheet_name='Appx B Baseline Criteria')

    if default_ws is not None and default_ws.title == 'Sheet' and default_ws is not ws:
        wb.remove(default_ws)

print(f'Exported: {export_path}')
print(f'First tab "Report Tables" holds the 6.2.1 / 6.2.3 / 6.2.4 + variance tables.')

judge_usage = usage_tracker.to_dict(agent_model=JUDGE_MODEL, judge_model=JUDGE_MODEL)
print(f"\nJudging cost this session ({JUDGE_MODEL}): ${judge_usage['judge_cost_usd']:.4f} "
      f"({judge_usage['judge_tokens_in']:,} in / {judge_usage['judge_tokens_out']:,} out)")